# LLM Engineering — Implementations

Two of the three algorithms gain a torch lane: LoRA because a frozen weight plus trainable low-rank factors is exactly what autograd is for, and DPO because `F.logsigmoid` is the stable form of the loss the notebook derives. BPE gets no authored lane at all: it is a purely discrete string algorithm — nothing to differentiate, so no torch lane — and the sandbox ships no `tokenizers` package, so no honest library comparison exists either; its NumPy lane stands alone. Neither `peft` (LoRA) nor `trl` (DPO) is available, so those two have no library lane and the torch lanes carry the comparison.

## 21_bpe_tokenizer

Merge the most frequent pair, repeat. *No authored lanes:* BPE is a purely discrete string algorithm with nothing to differentiate, and the sandbox has no `tokenizers` package, so neither a torch nor an honest library lane exists — the NumPy lane stands alone.

## 21_lora_linear

A frozen weight plus a low-rank correction. *No library lane:* `peft` is not in the sandbox, and rebuilding its adapter plumbing here would just be the torch lane with extra steps.

### torch

The same class on tensors: `W0` is drawn with the notebook's NumPy rng and simply left out of the graph — freezing is the absence of `requires_grad` — while `A` and `B` train. **What torch adds:** autograd delivers `dL/dA` and `dL/dB` for free, and shows the classic LoRA fact that with `B = 0` the gradient of `A` is exactly zero, so `B` has to move first.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Draw W0 and A from the notebook's global NumPy rng, then as_tensor them.
# 2. Only A and B get requires_grad_(True); the frozen W0 never sees a gradient.
# 3. With B = 0 the A-gradient is exactly zero - B must move before A can learn.
# 4. Step inside torch.no_grad() and clear .grad, like any hand-rolled torch loop.


class LoRALinear:
    """The same adapter on tensors: a frozen base weight, trainable A and B,
    and autograd producing the gradients the NumPy lane would derive by hand."""

    def __init__(self, in_features, out_features, r=8, alpha=16):
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Base frozen weights: identical NumPy draws, and no requires_grad —
        # freezing in torch is nothing more than leaving a tensor out of the graph.
        self.W0 = torch.as_tensor(rng.normal(0, 0.02, (in_features, out_features)))

        # LoRA weights: A is Kaiming init, B is zeros; only these two train.
        self.A = torch.as_tensor(
            rng.normal(0, np.sqrt(2.0 / in_features), (r, out_features))
        ).requires_grad_(True)
        self.B = torch.zeros((in_features, r), dtype=torch.float64, requires_grad=True)

    def forward(self, x):
        if not torch.is_tensor(x):
            x = torch.as_tensor(np.asarray(x, dtype=float))
        # Standard path
        base_out = x @ self.W0
        # LoRA path: x @ B @ A, scaled by alpha/r
        lora_out = (x @ self.B) @ self.A
        return base_out + self.scaling * lora_out


In [ ]:
# exports: y0, loss0, grad_A, grad_B, y1
rng = np.random.default_rng(2101)
_lora = LoRALinear(6, 4, r=2, alpha=8)
_X_eq = rng.normal(size=(3, 6))
_T_eq = rng.normal(size=(3, 4))

_y0_t = _lora.forward(_X_eq)
_loss_t = torch.mean((_y0_t - torch.as_tensor(_T_eq)) ** 2)
_loss_t.backward()

y0 = _y0_t.detach().numpy()
loss0 = float(_loss_t)
grad_A = _lora.A.grad.detach().numpy().copy()
grad_B = _lora.B.grad.detach().numpy().copy()

with torch.no_grad():
    _lora.A -= 0.01 * _lora.A.grad
    _lora.B -= 0.01 * _lora.B.grad
_lora.A.grad = None
_lora.B.grad = None

y1 = _lora.forward(_X_eq).detach().numpy()
print("loss at init:", round(loss0, 6))
print("max |grad_A| at init:", float(np.max(np.abs(grad_A))))


In [ ]:
# Frozen means frozen: the base weight never entered the graph as a parameter.
assert _lora.W0.grad is None, "backward must not touch the frozen W0"
# B = 0 gates the A path, so autograd's dL/dA at init is exactly zero — and
# therefore the SGD step above left A untouched.
assert float(np.max(np.abs(grad_A))) == 0.0, "with B=0, dL/dA = scaling*(xB)^T @ delta = 0"
# Autograd agrees with the pencil-and-paper dL/dB = scaling * x^T (delta A^T);
# _lora.A is still the init A because its gradient was zero.
_delta_eq = 2.0 * (torch.as_tensor(y0) - torch.as_tensor(_T_eq)) / y0.size
_gB_hand = _lora.scaling * torch.as_tensor(_X_eq).T @ (_delta_eq @ _lora.A.T)
assert float(torch.max(torch.abs(_gB_hand - torch.as_tensor(grad_B)))) < 1e-12
# One step through B activates the adapter and lowers the loss on this batch.
assert float(np.max(np.abs(y1 - y0))) > 0.0, "the LoRA path must switch on after one step"
_loss1_eq = float(torch.mean((_lora.forward(_X_eq) - torch.as_tensor(_T_eq)) ** 2))
assert _loss1_eq < loss0, "one SGD step on A, B reduces the training loss"
# Deployment property: the adapter merges into the base weight exactly.
_W_merged = (_lora.W0 + _lora.scaling * (_lora.B @ _lora.A)).detach()
_y_merged = torch.as_tensor(_X_eq) @ _W_merged
assert float(torch.max(torch.abs(_y_merged - _lora.forward(_X_eq)))) < 1e-12


## 21_dpo_loss

Preference optimisation as a logistic loss on implicit rewards. *No library lane:* `trl` is not in the sandbox, so the torch lane is the honest comparison.

### torch

The same loss with `-F.logsigmoid(margin)` in place of `-log(sigmoid(margin) + 1e-8)`: exact where the NumPy lane needed a fudge term, and stable at any margin. **What torch adds:** one `backward()` yields the per-response gradients, and the checks confirm autograd matches the pencil-and-paper `-(beta/n)*sigmoid(-margin)`.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

# hints:
# 1. F.logsigmoid(margin) is the stable form of log(sigmoid(margin)) - no 1e-8 fudge.
# 2. Mark only the policy log-probs requires_grad; the reference model is frozen.
# 3. One backward() yields d loss / d log-prob for every response in the batch.
# 4. The hand gradient is -(beta/n)*sigmoid(-margin) - check autograd against it.


def sigmoid(x):
    """Same map on tensors; the gradient check below reuses it."""
    return 1 / (1 + torch.exp(-x))


def dpo_loss(pi_theta_w, pi_ref_w, pi_theta_l, pi_ref_l, beta=0.1):
    """DPO loss on tensors. -F.logsigmoid(margin) computes -log(sigmoid(margin))
    exactly, where the NumPy lane needed a +1e-8 fudge inside the log."""
    # Implicit rewards
    r_theta_w = beta * (pi_theta_w - pi_ref_w)
    r_theta_l = beta * (pi_theta_l - pi_ref_l)

    # Margin
    margin = r_theta_w - r_theta_l

    # Loss is negative log sigmoid of the margin, in its stable form
    loss = -F.logsigmoid(margin)
    return torch.mean(loss)


In [ ]:
# exports: loss, margins, grad_ptw, grad_ptl
_pi_theta_w = torch.tensor([-12.3, -8.7, -15.2, -10.1], dtype=torch.float64,
                           requires_grad=True)
_pi_ref_w = torch.tensor([-13.0, -9.5, -14.8, -11.0], dtype=torch.float64)
_pi_theta_l = torch.tensor([-14.1, -11.2, -13.9, -12.5], dtype=torch.float64,
                           requires_grad=True)
_pi_ref_l = torch.tensor([-13.5, -10.0, -14.6, -11.8], dtype=torch.float64)

_loss_t = dpo_loss(_pi_theta_w, _pi_ref_w, _pi_theta_l, _pi_ref_l, beta=0.1)
_loss_t.backward()

loss = float(_loss_t)
margins = (0.1 * ((_pi_theta_w - _pi_ref_w)
                  - (_pi_theta_l - _pi_ref_l))).detach().numpy()
grad_ptw = _pi_theta_w.grad.numpy().copy()
grad_ptl = _pi_theta_l.grad.numpy().copy()
print("DPO loss:", round(loss, 6))


In [ ]:
# Autograd reproduces the pencil-and-paper gradient -(beta/n)*sigmoid(-margin).
_g_hand = (-(0.1 / 4) * sigmoid(torch.as_tensor(-margins))).numpy()
assert float(np.max(np.abs(_g_hand - grad_ptw))) < 1e-12
# Winner and loser tug with equal and opposite force on every pair.
assert float(np.max(np.abs(grad_ptw + grad_ptl))) < 1e-15
assert bool(np.all(grad_ptw < 0)), "raising a winner's log-prob always lowers the loss"
# Policy == reference gives margin 0 everywhere, so the loss is exactly log 2.
_l0_eq = float(dpo_loss(_pi_ref_w, _pi_ref_w, _pi_ref_l, _pi_ref_l, beta=0.1))
assert abs(_l0_eq - float(np.log(2.0))) < 1e-12
# Only log-ratios matter: shifting policy and reference together changes nothing.
_shift_eq = float(dpo_loss(_pi_theta_w.detach() + 5.0, _pi_ref_w + 5.0,
                           _pi_theta_l.detach() + 5.0, _pi_ref_l + 5.0, beta=0.1))
assert abs(_shift_eq - loss) < 1e-9
